In [ ]:
import streamlit as st
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
from PIL import UnidentifiedImageError

# ==============================
# Configuration
# ==============================
MODEL_PATH = "palm_ripeness_model.h5"
IMG_SIZE = (224, 224)
CLASS_NAMES = ["Overripe", "Ripe", "Unripe"]

CONFIDENCE_THRESHOLD = 85.0   # Minimum confidence to accept prediction
ENTROPY_THRESHOLD = 1.0       # Uncertainty threshold

# ==============================
# Load Model (Cached)
# ==============================
@st.cache_resource
def load_model():
    return tf.keras.models.load_model(MODEL_PATH)

model = load_model()

# ==============================
# UI
# ==============================
st.title("🌴 Palm Oil Fruit Ripeness Classifier")
st.write(
    "Upload an image of a **palm oil fruit** to classify it as "
    "**Unripe**, **Ripe**, or **Overripe**."
)

uploaded_file = st.file_uploader(
    "Choose an image (JPG, JPEG, PNG only)",
    type=["jpg", "jpeg", "png"]
)

# ==============================
# Image Processing & Prediction
# ==============================
if uploaded_file is not None:

    # --- Validate image file ---
    try:
        img = image.load_img(uploaded_file, target_size=IMG_SIZE)
    except UnidentifiedImageError:
        st.error("❌ Unsupported or corrupted image file.")
        st.stop()
    except Exception:
        st.error("❌ Unable to process the uploaded image.")
        st.stop()

    # Display image
    st.image(img, caption="Uploaded Image", use_container_width=True)

    # Preprocess
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0

    # Predict
    prediction = model.predict(img_array)
    label_index = np.argmax(prediction)
    confidence = prediction[0][label_index] * 100

    # Compute entropy (uncertainty)
    entropy = -np.sum(prediction * np.log(prediction + 1e-10))

    # --- Reject uncertain / non-palm images ---
    if confidence < CONFIDENCE_THRESHOLD or entropy > ENTROPY_THRESHOLD:
        st.warning(
            "⚠️ The image cannot be confidently identified as a palm oil fruit.\n\n"
            "Please upload a **clear palm oil fruit image**."
        )
        st.subheader("Model Confidence (for reference):")
        for i, class_name in enumerate(CLASS_NAMES):
            st.write(f"{class_name}: {prediction[0][i] * 100:.2f}%")
        st.stop()

    # ==============================
    # Output Prediction
    # ==============================
    st.subheader(f"✅ Prediction: **{CLASS_NAMES[label_index]}**")
    st.write(f"Confidence: **{confidence:.2f}%**")

    st.subheader("Class Probabilities:")
    for i, class_name in enumerate(CLASS_NAMES):
        st.write(f"{class_name}: {prediction[0][i] * 100:.2f}%")
